## 20. نقشه بازار

برای فروش آپارتمان مسکونی، حداقل موارد زیر بررسی شوند:

- میانه قیمت پیشنهادی هر مترمربع به تفکیک شهر
- میانه قیمت پیشنهادی هر مترمربع به تفکیک محله
- تعداد آگهی معتبر هر منطقه
- IQR یا شاخص پراکندگی
- درصد داده حذف‌شده در هر منطقه
- گران‌ترین و ارزان‌ترین محله قابل اعتماد

شهرهای مورد انتظار در تحلیل اصلی:

- تهران
- مشهد
- کرج
- اصفهان

### شرط رتبه‌بندی محله

یک محله صرفاً به‌دلیل داشتن میانه بالا یا پایین نباید رتبه‌بندی شود. تیم باید حداقل تعداد آگهی،
پوشش زمانی و کیفیت داده را کنترل کند.

ایمپورتها و خواندن فایل قبلی

In [60]:
import pandas as pd
import numpy as np

In [61]:
#1
df = pd.read_feather("C:/Outputs/02_df.feather")

c:\Users\Asus\anaconda3\Lib\site-packages\pandas\io\feather_format.py:124: FutureWarning: pyarrow.feather.read_feather is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  return feather.read_feather(


In [62]:
print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (1000000, 62)
['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug', 'created_at_month', 'user_type', 'description', 'title', 'rent_mode', 'rent_value', 'rent_to_single', 'rent_type', 'price_mode', 'price_value', 'credit_mode', 'credit_value', 'rent_credit_transform', 'transformable_price', 'transformable_credit', 'transformed_credit', 'transformable_rent', 'transformed_rent', 'land_size', 'building_size', 'deed_type', 'has_business_deed', 'floor', 'rooms_count', 'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator', 'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt', 'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas', 'has_heating_system', 'has_cooling_system', 'has_restroom', 'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool', 'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type', 'regular_person_capacity', 'extra_person_capacity', 'cost_per_extra_person', 'rent_price_on_regular_days', 'rent_pri

ساخت دیتای فروش آپارتمان مسکونی

In [63]:
#2
sell_df = df[
    df["cat2_slug"] == "residential-sell"
].copy()

print("Residential sale listings:", len(sell_df))

Residential sale listings: 558708


محاسبه قیمت پیشنهادی هر متر مربع

In [64]:
#3
sell_df["price_per_sqm"] = (
        sell_df["price_value"] /
        sell_df["building_size"]
    )

print(
    sell_df["price_per_sqm"]
    .describe()
)

count    5.349450e+05
mean     1.189154e+08
std      4.674581e+09
min      0.000000e+00
25%      1.282051e+07
50%      2.722222e+07
75%      4.814815e+07
max      1.587302e+12
Name: price_per_sqm, dtype: float64


ساخت دیتای اولیه برای تحلیل(اینحا فقط قیمت و مساحت مثبت را وارد محاسبات میکنیم)

In [65]:
#4
analysis_df = sell_df[
    (sell_df["price_value"] > 0) &
    (sell_df["building_size"] > 0) &
    (sell_df["price_per_sqm"] > 0)
].copy()

print("Initial valid records:", len(analysis_df))

Initial valid records: 533242


تعیین پرت ها

In [66]:
#5
LOWER_PRICE_LIMIT = 1_000_000

p99 = analysis_df["price_per_sqm"].quantile(0.99)

analysis_df["invalid_ppsqm_low_flag"] = (
    analysis_df["price_per_sqm"] < LOWER_PRICE_LIMIT
)

analysis_df["extreme_ppsqm_high_flag"] = (
    analysis_df["price_per_sqm"] > p99
)

print("P99:", p99)

print(
    "Low price outliers:",
    analysis_df["invalid_ppsqm_low_flag"].sum()
)

print(
    "High price outliers:",
    analysis_df["extreme_ppsqm_high_flag"].sum()
)

P99: 300000000.0
Low price outliers: 30165
High price outliers: 5192


ساخت دیتافریم جدید با داده های قابل اتکا

In [67]:
#6
analysis_valid = analysis_df[
    (analysis_df["price_per_sqm"] >= LOWER_PRICE_LIMIT) &
    (analysis_df["price_per_sqm"] <= p99)
].copy()

print("Analysis-valid records:", len(analysis_valid))

Analysis-valid records: 497885


ساخت جدول محله ها
/ از همه آگهی ها فروش استفاده میکنیم اما قیمت هارا فقط از analysis_valid میگیریم

In [68]:
#7
neighborhood_base = (
    sell_df[
        sell_df["city_slug"].notna() &
        sell_df["neighborhood_slug"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )
    .agg(
        total_listing_count=("price_value", "size"),
        missing_price_count=("price_value", lambda x: x.isna().sum())
    )
    .reset_index()
)

neighborhood_base["missing_price_rate"] = (
    neighborhood_base["missing_price_count"] /
    neighborhood_base["total_listing_count"]
)

neighborhood_base.head()

,city_slug,neighborhood_slug,total_listing_count,missing_price_count,missing_price_rate
0,ahvaz,amaniyeh-ahvaz,68,2,0.029412
1,ahvaz,ariyashahr,49,3,0.061224
2,ahvaz,baharestan-ahvaz,277,1,0.003610
3,ahvaz,bahonar,475,2,0.004211
4,ahvaz,camplojonoobi,176,1,0.005682


آمار قیمت برای هر محله

In [69]:
#8
neighborhood_price_summary = (
    analysis_valid[
        analysis_valid["city_slug"].notna() &
        analysis_valid["neighborhood_slug"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )["price_per_sqm"]
    .agg(
        valid_listing_count="count",
        median_price_per_sqm="median",
        p25_price_per_sqm=lambda x: x.quantile(0.25),
        p75_price_per_sqm=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

neighborhood_price_summary.head()

,city_slug,neighborhood_slug,valid_listing_count,median_price_per_sqm,p25_price_per_sqm,p75_price_per_sqm
0,ahvaz,amaniyeh-ahvaz,60,2.225000e+07,9.596946e+06,4.299191e+07
1,ahvaz,ariyashahr,42,2.993056e+07,2.000000e+07,3.373308e+07
2,ahvaz,baharestan-ahvaz,260,2.074603e+07,1.786012e+07,2.538215e+07
3,ahvaz,bahonar,466,2.731959e+07,2.328571e+07,3.166667e+07
4,ahvaz,camplojonoobi,155,2.125000e+07,1.500000e+07,2.780814e+07


محاسیه IQR

In [70]:
#9
neighborhood_price_summary["iqr_price_per_sqm"] = (
    neighborhood_price_summary["p75_price_per_sqm"] -
    neighborhood_price_summary["p25_price_per_sqm"]
)

نرخ اوتلایر برای هر محله

In [71]:
#10
neighborhood_outliers = (
    analysis_df[
        analysis_df["city_slug"].notna() &
        analysis_df["neighborhood_slug"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )
    .agg(
        price_valid_before_outlier_filter=("price_per_sqm", "count"),
        outlier_count=(
            "price_per_sqm",
            lambda x: (
                (x < LOWER_PRICE_LIMIT) |
                (x > p99)
            ).sum()
        )
    )
    .reset_index()
)

neighborhood_outliers["outlier_rate"] = (
    neighborhood_outliers["outlier_count"] /
    neighborhood_outliers["price_valid_before_outlier_filter"]
)

neighborhood_outliers.head()

,city_slug,neighborhood_slug,price_valid_before_outlier_filter,outlier_count,outlier_rate
0,ahvaz,amaniyeh-ahvaz,66,6,0.090909
1,ahvaz,ariyashahr,46,4,0.086957
2,ahvaz,baharestan-ahvaz,273,13,0.047619
3,ahvaz,bahonar,473,7,0.014799
4,ahvaz,camplojonoobi,172,17,0.098837


بررسی پوشش زمانی

In [72]:
#11
sell_df["created_at_month"] = pd.to_datetime(
    sell_df["created_at_month"],
    errors="coerce"
)

analysis_valid["created_at_month"] = pd.to_datetime(
    analysis_valid["created_at_month"],
    errors="coerce"
)

total_months = (
    sell_df["created_at_month"]
    .dropna()
    .nunique()
)

print("Total months in dataset:", total_months)

Total months in dataset: 43


برای هر محله تعداد ماه هایی که واقعا آگهی معتبر داشته اند را محاسبه میکنیم.

In [73]:
neighborhood_time = (
    analysis_valid[
        analysis_valid["city_slug"].notna() &
        analysis_valid["neighborhood_slug"].notna() &
        analysis_valid["created_at_month"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )["created_at_month"]
    .nunique()
    .rename("month_count")
    .reset_index()
)

neighborhood_time["time_coverage_rate"] = (
    neighborhood_time["month_count"] /
    total_months
)

neighborhood_time.head()

,city_slug,neighborhood_slug,month_count,time_coverage_rate
0,ahvaz,amaniyeh-ahvaz,8,0.186047
1,ahvaz,ariyashahr,8,0.186047
2,ahvaz,baharestan-ahvaz,10,0.232558
3,ahvaz,bahonar,9,0.209302
4,ahvaz,camplojonoobi,8,0.186047


ساخت neighborhood_market_summary/ تمام جدول ها را به هم وصل میکنیم

In [74]:
#12
neighborhood_market_summary = (
    neighborhood_base
    .merge(
        neighborhood_price_summary,
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
    .merge(
        neighborhood_outliers[
            [
                "city_slug",
                "neighborhood_slug",
                "outlier_rate"
            ]
        ],
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
    .merge(
        neighborhood_time,
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
)

neighborhood_market_summary.head()

,city_slug,neighborhood_slug,total_listing_count,missing_price_count,missing_price_rate,valid_listing_count,median_price_per_sqm,p25_price_per_sqm,p75_price_per_sqm,iqr_price_per_sqm,outlier_rate,month_count,time_coverage_rate
0,ahvaz,amaniyeh-ahvaz,68,2,0.029412,60.0,2.225000e+07,9.596946e+06,4.299191e+07,3.339497e+07,0.090909,8.0,0.186047
1,ahvaz,ariyashahr,49,3,0.061224,42.0,2.993056e+07,2.000000e+07,3.373308e+07,1.373308e+07,0.086957,8.0,0.186047
2,ahvaz,baharestan-ahvaz,277,1,0.003610,260.0,2.074603e+07,1.786012e+07,2.538215e+07,7.522035e+06,0.047619,10.0,0.232558
3,ahvaz,bahonar,475,2,0.004211,466.0,2.731959e+07,2.328571e+07,3.166667e+07,8.380952e+06,0.014799,9.0,0.209302
4,ahvaz,camplojonoobi,176,1,0.005682,155.0,2.125000e+07,1.500000e+07,2.780814e+07,1.280814e+07,0.098837,8.0,0.186047


پر کردن مواردی واقعا صفر هستند/ اینجا صفر منطقی است

In [75]:
neighborhood_market_summary["valid_listing_count"] = (
    neighborhood_market_summary["valid_listing_count"]
    .fillna(0)
    .astype(int)
)

neighborhood_market_summary["month_count"] = (
    neighborhood_market_summary["month_count"]
    .fillna(0)
    .astype(int)
)

neighborhood_market_summary["time_coverage_rate"] = (
    neighborhood_market_summary["time_coverage_rate"]
    .fillna(0)
)

neighborhood_market_summary["outlier_rate"] = (
    neighborhood_market_summary["outlier_rate"]
    .fillna(0)
)

تعریف Reliability

حداقل 50 آگهی معتبر/ حداقل 3 ماه پوشش زمانی/ حداقل 50درصد پوشش زمانی/ نرخ داده میسینگ حداکثر 30 درصد/ نرخ پرت حداکثر 10درصد

In [76]:
#14
MIN_VALID_LISTINGS = 50
MIN_MONTHS = 3
MIN_TIME_COVERAGE = 0.50
MAX_MISSING_PRICE_RATE = 0.30
MAX_OUTLIER_RATE = 0.10

neighborhood_market_summary["reliability_flag"] = np.where(
    (
        (neighborhood_market_summary["valid_listing_count"] >= MIN_VALID_LISTINGS) &
        (neighborhood_market_summary["month_count"] >= MIN_MONTHS) &
        (neighborhood_market_summary["time_coverage_rate"] >= MIN_TIME_COVERAGE) &
        (neighborhood_market_summary["missing_price_rate"] <= MAX_MISSING_PRICE_RATE) &
        (neighborhood_market_summary["outlier_rate"] <= MAX_OUTLIER_RATE)
    ),
    "Reliable",
    "Low_reliability"
)

فقط شهرهای اصلی پروژه

In [77]:
#15
target_cities = [
    "tehran",
    "mashhad",
    "karaj",
    "isfahan"
]

neighborhood_market_summary = (
    neighborhood_market_summary[
        neighborhood_market_summary["city_slug"].isin(target_cities)
    ]
    .copy()
)

print(
    neighborhood_market_summary["city_slug"]
    .value_counts()
)

city_slug
tehran            345
isfahan           200
mashhad           149
karaj              86
neka                0
                 ... 
garmsar             0
garmdareh           0
galugah-babol       0
galougah-babol      0
ziyabar             0
Name: count, Length: 421, dtype: int64


مرتب سازی نهایی ستون ها

In [78]:
#16
neighborhood_market_summary = neighborhood_market_summary[
    [
        "city_slug",
        "neighborhood_slug",
        "total_listing_count",
        "valid_listing_count",
        "median_price_per_sqm",
        "p25_price_per_sqm",
        "p75_price_per_sqm",
        "iqr_price_per_sqm",
        "missing_price_rate",
        "outlier_rate",
        "month_count",
        "time_coverage_rate",
        "reliability_flag"
    ]
].sort_values(
    ["city_slug", "median_price_per_sqm"],
    ascending=[True, False]
)

neighborhood_market_summary.head(20)

,city_slug,neighborhood_slug,total_listing_count,valid_listing_count,median_price_per_sqm,p25_price_per_sqm,p75_price_per_sqm,iqr_price_per_sqm,missing_price_rate,outlier_rate,month_count,time_coverage_rate,reliability_flag
233,isfahan,saadat-abad-isfahan,47,45,1.238095e+08,8.573718e+07,1.658031e+08,8.006593e+07,0.021277,0.021739,9,0.209302,Low_reliability
201,isfahan,mardavich,170,150,1.199138e+08,8.673913e+07,1.731828e+08,8.644367e+07,0.041176,0.074074,12,0.279070,Low_reliability
78,isfahan,abbasabad,54,51,1.062500e+08,6.937500e+07,1.258630e+08,5.648805e+07,0.037037,0.019231,9,0.209302,Low_reliability
83,isfahan,aineh-khaneh,69,59,1.000000e+08,5.000000e+07,1.362500e+08,8.625000e+07,0.014493,0.132353,9,0.209302,Low_reliability
101,isfahan,bagh-zereshk,56,53,1.000000e+08,7.200000e+07,1.500000e+08,7.800000e+07,0.035714,0.018519,8,0.186047,Low_reliability
186,isfahan,kouleh-parcheh,22,22,9.750000e+07,6.388889e+07,1.198295e+08,5.594066e+07,0.000000,0.000000,8,0.186047,Low_reliability
104,isfahan,bahar-azadi,55,52,9.401786e+07,7.182550e+07,1.352496e+08,6.342406e+07,0.036364,0.018868,9,0.209302,Low_reliability
100,isfahan,bagh-negar,33,33,8.857143e+07,7.000000e+07,1.235294e+08,5.352941e+07,0.000000,0.000000,8,0.186047,Low_reliability
203,isfahan,mehr-abad,96,94,8.776230e+07,6.583038e+07,1.303261e+08,6.449570e+07,0.010417,0.010526,10,0.232558,Low_reliability
114,isfahan,bishe-habib,26,23,8.571429e+07,7.400000e+07,1.315561e+08,5.755612e+07,0.076923,0.041667,8,0.186047,Low_reliability


بررسی کیفیت جدول

In [79]:
#17
print("Shape:")
print(neighborhood_market_summary.shape)

print("\nReliability:")
print(
    neighborhood_market_summary["reliability_flag"]
    .value_counts()
)

print("\nMissing values:")
print(
    neighborhood_market_summary.isna().sum()
)

print("\nTarget cities:")
print(
    neighborhood_market_summary["city_slug"]
    .value_counts()
)

Shape:
(780, 13)

Reliability:
reliability_flag
Low_reliability    780
Name: count, dtype: int64

Missing values:
city_slug               0
neighborhood_slug       0
total_listing_count     0
valid_listing_count     0
median_price_per_sqm    0
p25_price_per_sqm       0
p75_price_per_sqm       0
iqr_price_per_sqm       0
missing_price_rate      0
outlier_rate            0
month_count             0
time_coverage_rate      0
reliability_flag        0
dtype: int64

Target cities:
city_slug
tehran            345
isfahan           200
mashhad           149
karaj              86
neka                0
                 ... 
garmsar             0
garmdareh           0
galugah-babol       0
galougah-babol      0
ziyabar             0
Name: count, Length: 421, dtype: int64


اصلاح پوشش زمانی

In [80]:
#17.5 
# The dataset contains a small number of records before May 2024
# and after December 2024.
# From May 2024 onward, listing volume becomes substantially larger
# and more stable.
#
# Therefore, for neighborhood-level market mapping and reliability
# assessment, we use a six-month high-coverage window:
# May 2024 to October 2024.

project_months = pd.period_range(
    start="2024-05",
    end="2024-10",
    freq="M"
)

print("Selected analysis months:")
print(project_months)

Selected analysis months:
PeriodIndex(['2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10'], dtype='period[M]')


In [81]:
#17.6
sell_project = sell_df[
    sell_df["created_at_month"]
    .dt.to_period("M")
    .isin(project_months)
].copy()

analysis_valid_project = analysis_valid[
    analysis_valid["created_at_month"]
    .dt.to_period("M")
    .isin(project_months)
].copy()

print("Total residential-sale listings in selected period:")
print(len(sell_project))

print("\nValid price listings in selected period:")
print(len(analysis_valid_project))

print("\nMonthly listing counts:")
print(
    sell_project["created_at_month"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

Total residential-sale listings in selected period:
405674

Valid price listings in selected period:
361193

Monthly listing counts:
created_at_month
2024-05    66688
2024-06    66789
2024-07    70039
2024-08    68674
2024-09    63075
2024-10    70409
Freq: M, Name: count, dtype: int64


In [82]:
#17.7
total_months = len(project_months)

neighborhood_time = (
    analysis_valid_project[
        analysis_valid_project["city_slug"].notna() &
        analysis_valid_project["neighborhood_slug"].notna() &
        analysis_valid_project["created_at_month"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )["created_at_month"]
    .nunique()
    .rename("month_count")
    .reset_index()
)

neighborhood_time["time_coverage_rate"] = (
    neighborhood_time["month_count"] /
    total_months
)

neighborhood_time.head()

,city_slug,neighborhood_slug,month_count,time_coverage_rate
0,ahvaz,amaniyeh-ahvaz,6,1.0
1,ahvaz,ariyashahr,6,1.0
2,ahvaz,baharestan-ahvaz,6,1.0
3,ahvaz,bahonar,6,1.0
4,ahvaz,camplojonoobi,6,1.0


ساخت دوباره بیس با تایم پوشش زمانی اصلاح شده

In [83]:
#18
neighborhood_base = (
    sell_project[
        sell_project["city_slug"].notna() &
        sell_project["neighborhood_slug"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )
    .agg(
        total_listing_count=("price_value", "size"),
        missing_price_count=(
            "price_value",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

neighborhood_base["missing_price_rate"] = (
    neighborhood_base["missing_price_count"] /
    neighborhood_base["total_listing_count"]
)

آمار قیمت

In [84]:
#19
neighborhood_price_summary = (
    analysis_valid_project[
        analysis_valid_project["city_slug"].notna() &
        analysis_valid_project["neighborhood_slug"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )["price_per_sqm"]
    .agg(
        valid_listing_count="count",
        median_price_per_sqm="median",
        p25_price_per_sqm=lambda x: x.quantile(0.25),
        p75_price_per_sqm=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

neighborhood_price_summary["iqr_price_per_sqm"] = (
    neighborhood_price_summary["p75_price_per_sqm"] -
    neighborhood_price_summary["p25_price_per_sqm"]
)

نرخ اوتلایر برحسب محله

In [85]:
#20
neighborhood_outliers = (
    analysis_df[
        analysis_df["city_slug"].notna() &
        analysis_df["neighborhood_slug"].notna() &
        analysis_df["created_at_month"].dt.to_period("M").isin(project_months)
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )
    .agg(
        price_valid_before_outlier_filter=(
            "price_per_sqm",
            "count"
        ),
        outlier_count=(
            "price_per_sqm",
            lambda x: (
                (x < LOWER_PRICE_LIMIT) |
                (x > p99)
            ).sum()
        )
    )
    .reset_index()
)

neighborhood_outliers["outlier_rate"] = (
    neighborhood_outliers["outlier_count"] /
    neighborhood_outliers["price_valid_before_outlier_filter"]
)

ساخت جدول نهایی

In [86]:
#21
neighborhood_market_summary = (
    neighborhood_base
    .merge(
        neighborhood_price_summary,
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
    .merge(
        neighborhood_outliers[
            [
                "city_slug",
                "neighborhood_slug",
                "outlier_rate"
            ]
        ],
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
    .merge(
        neighborhood_time,
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
)

neighborhood_market_summary["valid_listing_count"] = (
    neighborhood_market_summary["valid_listing_count"]
    .fillna(0)
    .astype(int)
)

neighborhood_market_summary["month_count"] = (
    neighborhood_market_summary["month_count"]
    .fillna(0)
    .astype(int)
)

neighborhood_market_summary["time_coverage_rate"] = (
    neighborhood_market_summary["time_coverage_rate"]
    .fillna(0)
)

neighborhood_market_summary["outlier_rate"] = (
    neighborhood_market_summary["outlier_rate"]
    .fillna(0)
)

Reliability نهایی

In [87]:
#22
MIN_VALID_LISTINGS = 50
MIN_MONTHS = 3
MIN_TIME_COVERAGE = 0.50
MAX_MISSING_PRICE_RATE = 0.30
MAX_OUTLIER_RATE = 0.10

neighborhood_market_summary["reliability_flag"] = np.where(
    (
        (neighborhood_market_summary["valid_listing_count"] >= MIN_VALID_LISTINGS) &
        (neighborhood_market_summary["month_count"] >= MIN_MONTHS) &
        (neighborhood_market_summary["time_coverage_rate"] >= MIN_TIME_COVERAGE) &
        (neighborhood_market_summary["missing_price_rate"] <= MAX_MISSING_PRICE_RATE) &
        (neighborhood_market_summary["outlier_rate"] <= MAX_OUTLIER_RATE)
    ),
    "Reliable",
    "Low_reliability"
)

فقط 4 شهر اصلی

In [88]:
#23
target_cities = [
    "tehran",
    "mashhad",
    "karaj",
    "isfahan"
]

neighborhood_market_summary = (
    neighborhood_market_summary[
        neighborhood_market_summary["city_slug"].isin(target_cities)
    ]
    .copy()
)

ترتیب ستون ها

In [89]:
#24
neighborhood_market_summary = neighborhood_market_summary[
    [
        "city_slug",
        "neighborhood_slug",
        "total_listing_count",
        "valid_listing_count",
        "median_price_per_sqm",
        "p25_price_per_sqm",
        "p75_price_per_sqm",
        "iqr_price_per_sqm",
        "missing_price_rate",
        "outlier_rate",
        "month_count",
        "time_coverage_rate",
        "reliability_flag"
    ]
].sort_values(
    ["city_slug", "median_price_per_sqm"],
    ascending=[True, False]
)

چک نهایی

In [90]:
#25
print("Shape:")
print(neighborhood_market_summary.shape)

print("\nReliability:")
print(
    neighborhood_market_summary["reliability_flag"]
    .value_counts()
)

print("\nMissing values:")
print(
    neighborhood_market_summary.isna().sum()
)

print("\nTarget cities:")
print(
    neighborhood_market_summary["city_slug"]
    .value_counts()
)

Shape:
(780, 13)

Reliability:
reliability_flag
Reliable           419
Low_reliability    361
Name: count, dtype: int64

Missing values:
city_slug               0
neighborhood_slug       0
total_listing_count     0
valid_listing_count     0
median_price_per_sqm    0
p25_price_per_sqm       0
p75_price_per_sqm       0
iqr_price_per_sqm       0
missing_price_rate      0
outlier_rate            0
month_count             0
time_coverage_rate      0
reliability_flag        0
dtype: int64

Target cities:
city_slug
tehran            345
isfahan           200
mashhad           149
karaj              86
neka                0
                 ... 
garmsar             0
garmdareh           0
galugah-babol       0
galougah-babol      0
ziyabar             0
Name: count, Length: 421, dtype: int64


بررسی منطقی Reliability

In [91]:
#26
reliability_check = neighborhood_market_summary.copy()

reliability_check["check_listing"] = (
    reliability_check["valid_listing_count"] >= MIN_VALID_LISTINGS
)

reliability_check["check_month"] = (
    reliability_check["month_count"] >= MIN_MONTHS
)

reliability_check["check_coverage"] = (
    reliability_check["time_coverage_rate"] >= MIN_TIME_COVERAGE
)

reliability_check["check_missing"] = (
    reliability_check["missing_price_rate"] <= MAX_MISSING_PRICE_RATE
)

reliability_check["check_outlier"] = (
    reliability_check["outlier_rate"] <= MAX_OUTLIER_RATE
)

print(
    reliability_check[
        [
            "check_listing",
            "check_month",
            "check_coverage",
            "check_missing",
            "check_outlier"
        ]
    ].apply(pd.Series.value_counts)
)

       check_listing  check_month  check_coverage  check_missing  \
True             462          756             756            779   
False            318           24              24              1   

       check_outlier  
True             684  
False             96  


Reliable neighborhoods

In [92]:
#27
reliable_neighborhoods = (
    neighborhood_market_summary[
        neighborhood_market_summary["reliability_flag"] == "Reliable"
    ]
    .copy()
)

print(
    "Reliable neighborhoods:",
    len(reliable_neighborhoods)
)

Reliable neighborhoods: 419


گرانترین و ارزانترین محله ها

In [93]:
#28
for city in target_cities:

    city_data = (
        reliable_neighborhoods[
            reliable_neighborhoods["city_slug"] == city
        ]
        .sort_values(
            "median_price_per_sqm",
            ascending=False
        )
    )

    print("\n" + "=" * 70)
    print(city.upper())
    print("=" * 70)

    if city_data.empty:
        print("No reliable neighborhood found.")
        continue

    print("\nMost expensive reliable neighborhood:")
    print(
        city_data[
            [
                "neighborhood_slug",
                "valid_listing_count",
                "median_price_per_sqm",
                "iqr_price_per_sqm",
                "missing_price_rate",
                "outlier_rate",
                "month_count",
                "time_coverage_rate"
            ]
        ].head(1)
    )

    print("\nCheapest reliable neighborhood:")
    print(
        city_data[
            [
                "neighborhood_slug",
                "valid_listing_count",
                "median_price_per_sqm",
                "iqr_price_per_sqm",
                "missing_price_rate",
                "outlier_rate",
                "month_count",
                "time_coverage_rate"
            ]
        ].tail(1)
    )


TEHRAN

Most expensive reliable neighborhood:
    neighborhood_slug  valid_listing_count  median_price_per_sqm  \
896            hekmat                  133          1.770833e+08   

     iqr_price_per_sqm  missing_price_rate  outlier_rate  month_count  \
896         55000000.0            0.006897      0.076389            6   

     time_coverage_rate  
896                 1.0  

Cheapest reliable neighborhood:
     neighborhood_slug  valid_listing_count  median_price_per_sqm  \
1057        sharifabad                  182          1.307895e+07   

      iqr_price_per_sqm  missing_price_rate  outlier_rate  month_count  \
1057       1.673905e+07            0.078341          0.09            6   

      time_coverage_rate  
1057                 1.0  

MASHHAD

Most expensive reliable neighborhood:
    neighborhood_slug  valid_listing_count  median_price_per_sqm  \
470       sadjadshahr                  350          8.903904e+07   

     iqr_price_per_sqm  missing_price_rate  outlier_rate 

10 محاه گران هر شهر

In [94]:
#29
top_expensive_neighborhoods = (
    reliable_neighborhoods
    .sort_values(
        ["city_slug", "median_price_per_sqm"],
        ascending=[True, False]
    )
    .groupby(
        "city_slug",
        observed=True
    )
    .head(10)
)

top_expensive_neighborhoods[
    [
        "city_slug",
        "neighborhood_slug",
        "valid_listing_count",
        "median_price_per_sqm",
        "iqr_price_per_sqm",
        "time_coverage_rate"
    ]
]

,city_slug,neighborhood_slug,valid_listing_count,median_price_per_sqm,iqr_price_per_sqm,time_coverage_rate
183,isfahan,mardavich,107,1.197368e+08,8.861546e+07,1.0
185,isfahan,mehr-abad,66,9.101010e+07,6.929394e+07,1.0
76,isfahan,azar-isfahan,60,7.961957e+07,5.744318e+07,1.0
184,isfahan,marnan,59,7.933333e+07,4.238685e+07,1.0
151,isfahan,jolfa,84,7.600000e+07,2.668159e+07,1.0
78,isfahan,bagh-daryacheh,104,7.441589e+07,4.283974e+07,1.0
239,isfahan,sheykh-sadough,149,7.407407e+07,3.785714e+07,1.0
100,isfahan,chaharbaghkhajoo,87,6.500000e+07,2.630736e+07,1.0
98,isfahan,bozorgmehr,211,6.250000e+07,2.873971e+07,1.0
103,isfahan,dashtestan,184,5.852361e+07,2.084490e+07,1.0


10 محله ارزان هر شهر

In [95]:
#30
top_cheap_neighborhoods = (
    reliable_neighborhoods
    .sort_values(
        ["city_slug", "median_price_per_sqm"],
        ascending=[True, True]
    )
    .groupby(
        "city_slug",
        observed=True
    )
    .head(10)
)

top_cheap_neighborhoods[
    [
        "city_slug",
        "neighborhood_slug",
        "valid_listing_count",
        "median_price_per_sqm",
        "iqr_price_per_sqm",
        "time_coverage_rate"
    ]
]

,city_slug,neighborhood_slug,valid_listing_count,median_price_per_sqm,iqr_price_per_sqm,time_coverage_rate
129,isfahan,ghale-nou,57,1.300000e+07,1.772222e+07,1.0
111,isfahan,dorcheh,111,1.500000e+07,2.192427e+07,1.0
146,isfahan,jarvakan,84,1.799867e+07,1.095225e+07,1.0
258,isfahan,zeynabieh,81,1.909091e+07,1.154167e+07,1.0
127,isfahan,ghaemiyeh,58,2.250000e+07,2.213791e+07,1.0
211,isfahan,rehnan,85,2.250000e+07,1.517391e+07,1.0
88,isfahan,baharestan-esfahan,1055,2.400990e+07,1.477497e+07,1.0
126,isfahan,gaz,166,2.479243e+07,5.819450e+06,1.0
59,isfahan,24-metri,319,2.751724e+07,9.306763e+06,1.0
167,isfahan,kojan,74,2.794737e+07,8.217424e+06,1.0


سیو خروجی

In [ ]:
#31
neighborhood_market_summary.to_feather(
    "C:/Outputs/market_map.feather"
)

print("Final market map saved successfully.")

خلاصه نهایی

In [96]:
#32
print(
    neighborhood_market_summary[
        [
            "city_slug",
            "neighborhood_slug",
            "valid_listing_count",
            "median_price_per_sqm",
            "iqr_price_per_sqm",
            "missing_price_rate",
            "outlier_rate",
            "month_count",
            "time_coverage_rate",
            "reliability_flag"
        ]
    ].head(20)
)

    city_slug       neighborhood_slug  valid_listing_count  \
123   isfahan                 fizadan                    2   
215   isfahan     saadat-abad-isfahan                   36   
183   isfahan               mardavich                  107   
60    isfahan               abbasabad                   37   
83    isfahan            bagh-zereshk                   39   
168   isfahan          kouleh-parcheh                   13   
86    isfahan             bahar-azadi                   34   
82    isfahan              bagh-negar                   19   
185   isfahan               mehr-abad                   66   
65    isfahan            aineh-khaneh                   45   
241   isfahan                  sichan                   32   
96    isfahan             bishe-habib                   18   
114   isfahan  emam-jafar-sadegh-town                    4   
101   isfahan                charkhab                   24   
76    isfahan            azar-isfahan                   60   
184   is

In [58]:
neighborhood_market_summary.columns

Index(['city_slug', 'neighborhood_slug', 'total_listing_count',
       'valid_listing_count', 'median_price_per_sqm', 'p25_price_per_sqm',
       'p75_price_per_sqm', 'iqr_price_per_sqm', 'missing_price_rate',
       'outlier_rate', 'month_count', 'time_coverage_rate',
       'reliability_flag'],
      dtype='object')

سیو فایل

In [97]:
neighborhood_market_summary.to_csv("market_map.csv")

In [ ]:
# Code

# محل پیاده‌سازی تیم:
# یک جدول neighborhood_market_summary بسازید.
#
# ستون‌های پیشنهادی:
# city
# neighborhood
# valid_listing_count
# median_price_per_sqm
# p25_price_per_sqm
# p75_price_per_sqm
# missing_price_rate
# outlier_rate
# reliability_flag